In [1]:
import os
import numpy as np
import pandas as pd
import pandas_profiling
import plotnine
from plotnine import *  # Provides a ggplot-like interface to matplotlib.
from IPython.display import display
from datetime import datetime
import subprocess
#!pip3 install --upgrade dsub

## Plot setup.
theme_set(theme_bw(base_size = 11)) # Default theme for plots.

def get_boxplot_fun_data(df):
  """Returns a data frame with a y position and a label, for use annotating ggplot boxplots.

  Args:
    d: A data frame.
  Returns:
    A data frame with column y as max and column label as length.
  """
  d = {'y': max(df), 'label': f'N = {len(df)}'}
  return(pd.DataFrame(data=d, index=[0]))

# NOTE: if you get any errors from this cell, restart your kernel and run it again.
my_bucket = os.getenv('WORKSPACE_BUCKET')


/opt/conda/lib/python3.10/site-packages/numba/core/decorators.py:262: NumbaDeprecationWarning: numba.generated_jit is deprecated. Please see the documentation at: https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-generated-jit for more information and advice on a suitable replacement.
  warnings.warn(msg, NumbaDeprecationWarning)
/opt/conda/lib/python3.10/site-packages/visions/backends/shared/nan_handling.py:51: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def hasna(x: np.ndarray) -> bool:
/tmp/ipykernel_249/2004277905.py:4: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profilin

In [4]:
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env USER_NAME={USER_NAME}

env: USER_NAME=micah_hysong


In [5]:
## MODIFY FOR FULL DATA RUN
# Use underscores, not whitespace since it will become part of the bucket path.
JOB_NAME="plink_for_PRS_construction"

# Save this Python variable as an environment variable so that its easier to use within %%bash cells.
%env JOB_NAME={JOB_NAME}

env: JOB_NAME=plink_for_PRS_construction


In [6]:
%%writefile ~/plink.sh
#!/bin/bash

set -o pipefail
set -o errexit

echo "Chromosome: ${chrom}"
echo "Output path is ${OUTPUT_PATH}"

# Define an array of disease names
diseases=("asthma" "atrial_fibrilation" "breast_cancer" "ckd" "coronary_heart_disease" "hypercholesterolemia"
          "prostate_cancer" "t1d" "t2d")

# Loop through each disease
for disease in "${diseases[@]}"; do
    echo "Processing Disease: ${disease}"
    
    # Extract relevant SNPs from weights file
    awk -v chrom="chr${chrom}" -v disease="${disease}" 'BEGIN { FS = " " ; OFS = " " } $3 == chrom && $8 == disease { print $1, $2, $3, $4, $5, $6, $7, $8 }' "$weights_file" > "subset_${disease}_chr${chrom}.txt"

    # Check if subset file is empty
    if [[ ! -s "subset_${disease}_chr${chrom}.txt" ]]; then
        echo "No valid SNPs for ${disease} on chromosome ${chrom}. Skipping..."
        continue  # Skip to the next disease in the loop
    fi

    # Construct PRS (1 = SNP ID, 2 = Effect Allele, 3 = Beta) 
    if ! plink --bfile "${bed_prefix}" \
        --score "subset_${disease}_chr${chrom}.txt" 7 5 2 header \
        --out "${OUT_PREFIX}_${disease}"; then
        echo "Plink failed for ${disease} on chromosome ${chrom}. Skipping..."
        continue  # Skip to the next disease in the loop
    fi

    # Move results to output path
    plink_results="${OUT_PREFIX}_${disease}.profile"
    if [[ -f "${plink_results}" ]]; then
        echo "Plink results: ${plink_results}"
        mv "${plink_results}" "${OUTPUT_PATH}/"
    else
        echo "No valid output for ${disease} on chromosome ${chrom}. Skipping..."
    fi
done




Writing /home/jupyter/plink.sh


In [7]:
!gsutil cp /home/jupyter/plink.sh ${WORKSPACE_BUCKET}/pathways/plink.sh

Copying file:///home/jupyter/plink.sh [Content-Type=text/x-sh]...
/ [1 files][  1.6 KiB/  1.6 KiB]                                                
Operation completed over 1 objects/1.6 KiB.                                      


In [8]:
name_of_file_in_bucket = 'combined_weights.txt'
os.system(f"gsutil cp '{my_bucket}/weights/hg38/{name_of_file_in_bucket}' .")
weights = pd.read_csv(name_of_file_in_bucket, sep=" ")
weights.head()

Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/combined_weights.txt...
| [1 files][448.5 MiB/448.5 MiB]                                                
Operation completed over 1 objects/448.5 MiB.                                    


,id,weight,chr,pos,a1,a2,SNPID,disease
0,chr1:843942:A:G,0.000017,chr1,843942,G,A,chr1:843942:G:A,asthma
1,chr1:1070426:C:T,-0.000277,chr1,1070426,T,C,chr1:1070426:T:C,asthma
2,chr1:1081817:C:T,-0.000066,chr1,1081817,C,T,chr1:1081817:C:T,asthma
3,chr1:1082207:C:T,-0.000139,chr1,1082207,T,C,chr1:1082207:T:C,asthma
4,chr1:1083324:A:G,-0.000159,chr1,1083324,A,G,chr1:1083324:A:G,asthma


In [9]:
results_folder = os.path.join(
    os.getenv('WORKSPACE_BUCKET'),
    'dsub',
    'plink')

results_folder

'gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink'

In [10]:
%env RESULTS_FOLDER={results_folder}

env: RESULTS_FOLDER=gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/plink


In [11]:
my_bucket

'gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6'

In [12]:
# %%bash

# chrom=1
# weight="asthma"
# gsutil cp "gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/combined_weights.txt" "./"
# weights_file="combined_weights.txt"
# OUTPUT_PATH="gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/data"

# echo "chr${chrom}"
# echo "Weight: ${weight}"
# echo "Output path is ${OUTPUT_PATH}"

# head $weights_file
    
# awk -v chrom="chr${chrom}" -v weight="${weight}" 'BEGIN { FS = " " ; OFS = " " } $3 == chrom && $8 == weight { print $1, $2, $3, $4, $5, $6, $7, $8 }' "$weights_file" > "subset_${weight}_chr${chrom}.txt"

# echo "new file: "

# head subset_${weight}_chr${chrom}.txt
    
# if [ -s "${OUTPUT_PATH}/subset_${weight}_chr${chrom}.txt" ]; then
#     echo "Score file is not empty. Running PLINK command..."

# else
#     echo "Score file is empty. Skipping PLINK command."
# fi

In [13]:
# %%bash --out "plink"

# # Get a shorter username to leave more characters for the job name.
# DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

# # For AoU RWB projects network name is "network".
# AOU_NETWORK=network
# AOU_SUBNETWORK=subnetwork

# # The biggest chromosomes lead to a crash. Trying again with a beefier machine type
# MACHINE_TYPE="n2-highmem-64"
# BASH_SCRIPT="${WORKSPACE_BUCKET}/pathways/plink.sh"

# # Max retry limit per chromosome
# MAX_RETRIES=3

# # Function to submit a job asynchronously
# submit_job() {
#     local chromo=$1
#     local attempt=$2

#     # Set disk size dynamically based on chromosome number
#     if [[ "$chromo" -eq 1 || "$chromo" -eq 2 ]]; then
#         DISK_SIZE=1000  # Extra large disk for chromosomes 1 & 2
#     elif [[ "$chromo" -le 14 ]]; then
#         DISK_SIZE=500  # Larger disk for chromosomes 3-14
#     else
#         DISK_SIZE=150  # Standard disk for chromosomes 15-22
#     fi

#     BED_INFIX="fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}"
    
#     echo "Submitting job for chromosome ${chromo}, attempt ${attempt}..."

#     dsub \
#         --provider google-cls-v2 \
#         --user-project "${GOOGLE_PROJECT}" \
#         --project "${GOOGLE_PROJECT}" \
#         --network "${AOU_NETWORK}" \
#         --subnetwork "${AOU_SUBNETWORK}" \
#         --service-account "$(gcloud config get-value account)" \
#         --user "${USER_NAME}" \
#         --regions us-central1 \
#         --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/{job-id}-{task-id}-{task-attempt}.log" \
#         --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.1.15" \
#         --name "${JOB_NAME}_chr${chromo}_attempt${attempt}" \
#         --preemptible \
#         --disk-size "${DISK_SIZE}" \
#         --boot-disk-size 150 \
#         --machine-type="${MACHINE_TYPE}" \
#         --script "${BASH_SCRIPT}" \
#         --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
#         --env bed_prefix="/mnt/data/input/gs/${BED_INFIX}" \
#         --input weights_file="gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/combined_weights.txt" \
#         --env chrom=${chromo} \
#         --input bed_file="gs://${BED_INFIX}.bed" \
#         --input bim_file="gs://${BED_INFIX}.bim" \
#         --input fam_file="gs://${BED_INFIX}.fam" \
#         --env OUT_PREFIX="PRS_chr${chromo}" \
#         --output-recursive OUTPUT_PATH="${RESULTS_FOLDER}" \
#         --wait &  # Run job in background to avoid blocking
# }

# # Track job attempts and running jobs
# declare -A job_attempts
# declare -A running_jobs

# # Submit jobs for each chromosome asynchronously
# for chromo in {2..12} 14 {17..21}; do
#     submit_job "$chromo" 1
#     job_attempts["$chromo"]=1  # Track initial attempt
#     running_jobs["$chromo"]=1  # Mark as running
# done

# # Monitor job statuses and retry failed ones independently
# while [[ ${#running_jobs[@]} -gt 0 ]]; do
#     sleep 1800  # Wait 30 minutes before checking statuses

#     for chromo in "${!running_jobs[@]}"; do
#         # Check if job is still running
#         dstat --provider google-cls-v2 --jobs "${JOB_NAME}_chr${chromo}_attempt${job_attempts[$chromo]}" --status RUNNING > /dev/null 2>&1
#         if [[ $? -eq 0 ]]; then
#             echo "Chromosome ${chromo} is still running. Checking again later."
#             continue  # Skip to the next chromosome without retrying
#         fi

#         # Check if the job for this chromosome failed
#         dstat --provider google-cls-v2 --jobs "${JOB_NAME}_chr${chromo}_attempt${job_attempts[$chromo]}" --status FAILURE > /dev/null 2>&1
#         if [[ $? -eq 0 ]]; then
#             echo "Chromosome ${chromo} failed on attempt ${job_attempts[$chromo]}. Retrying..."
            
#             # Increment attempt counter
#             ((job_attempts[$chromo]++))

#             # Retry job if under retry limit
#             if [[ ${job_attempts[$chromo]} -le $MAX_RETRIES ]]; then
#                 submit_job "$chromo" "${job_attempts[$chromo]}"
#             else
#                 echo "Chromosome ${chromo} failed after $MAX_RETRIES retries. Moving on."
#                 unset running_jobs["$chromo"]  # Remove from retry tracking
#             fi
#         else
#             # If job succeeded, remove from tracking
#             echo "Chromosome ${chromo} completed successfully."
#             unset running_jobs["$chromo"]
#         fi
#     done
# done

# echo "All jobs submitted, monitored, and retried independently. Check logs for final status."



In [14]:
%%bash --out "plink"

# Get a shorter username to leave more characters for the job name.
DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

# For AoU RWB projects network name is "network".
AOU_NETWORK=network
AOU_SUBNETWORK=subnetwork
MACHINE_TYPE="n2-highmem-64"
BASH_SCRIPT="${WORKSPACE_BUCKET}/pathways/plink.sh"

# Loop through selected chromosomes
for chromo in {2..12} {17..21}; do

    # Set disk size and machine type based on chromosome number
    if [[ "$chromo" -eq 1 || "$chromo" -eq 2 ]]; then
        DISK_SIZE=1000
    elif [[ "$chromo" -le 14 ]]; then
        DISK_SIZE=800
    else
        DISK_SIZE=500
    fi

    BED_INFIX="fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr${chromo}"

    echo "Launching job for chromosome ${chromo} with ${MACHINE_TYPE} and ${DISK_SIZE}GB disk."

    dsub \
    --provider google-cls-v2 \
    --user-project "${GOOGLE_PROJECT}" \
    --project "${GOOGLE_PROJECT}" \
    --network "${AOU_NETWORK}" \
    --subnetwork "${AOU_SUBNETWORK}" \
    --service-account "$(gcloud config get-value account)" \
    --user "${USER_NAME}" \
    --regions us-central1 \
    --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/{job-id}-{task-id}-{task-attempt}.log" \
    --image "us.gcr.io/broad-dsp-gcr-public/terra-jupyter-aou:2.1.15" \
    --name "${JOB_NAME}_chr${chromo}" \
    --disk-size "${DISK_SIZE}" \
    --boot-disk-size 150 \
    --machine-type="${MACHINE_TYPE}" \
    --script "${BASH_SCRIPT}" \
    --env GOOGLE_PROJECT=${GOOGLE_PROJECT} \
    --env bed_prefix="/mnt/data/input/gs/${BED_INFIX}" \
    --input weights_file="gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/weights/hg38/combined_weights.txt" \
    --env chrom=${chromo} \
    --input bed_file="gs://${BED_INFIX}.bed" \
    --input bim_file="gs://${BED_INFIX}.bim" \
    --input fam_file="gs://${BED_INFIX}.fam" \
    --env OUT_PREFIX="PRS_chr${chromo}" \
    --output-recursive OUTPUT_PATH="${RESULTS_FOLDER}"
done

Job properties:
  job-id: plink-for---micah-hysong--250324-231000-52
  job-name: plink-for-prs-construction-chr2
  user-id: micah-hysong
Provider internal-id (operation): projects/1017278029283/locations/us-central1/operations/7272389573703619657
Launched job-id: plink-for---micah-hysong--250324-231000-52
To check the status, run:
  dstat --provider google-cls-v2 --project terra-vpc-sc-482f9fa9 --location us-central1 --jobs 'plink-for---micah-hysong--250324-231000-52' --users 'micah-hysong' --status '*'
To cancel the job, run:
  ddel --provider google-cls-v2 --project terra-vpc-sc-482f9fa9 --location us-central1 --jobs 'plink-for---micah-hysong--250324-231000-52' --users 'micah-hysong'
Job properties:
  job-id: plink-for---micah-hysong--250324-231002-62
  job-name: plink-for-prs-construction-chr3
  user-id: micah-hysong
Provider internal-id (operation): projects/1017278029283/locations/us-central1/operations/8162431221667636218
Launched job-id: plink-for---micah-hysong--250324-231002-6

Job properties:
  job-id: plink-for---micah-hysong--250324-231024-41
  job-name: plink-for-prs-construction-chr18
  user-id: micah-hysong
Provider internal-id (operation): projects/1017278029283/locations/us-central1/operations/8260497591838350730
Launched job-id: plink-for---micah-hysong--250324-231024-41
To check the status, run:
  dstat --provider google-cls-v2 --project terra-vpc-sc-482f9fa9 --location us-central1 --jobs 'plink-for---micah-hysong--250324-231024-41' --users 'micah-hysong' --status '*'
To cancel the job, run:
  ddel --provider google-cls-v2 --project terra-vpc-sc-482f9fa9 --location us-central1 --jobs 'plink-for---micah-hysong--250324-231024-41' --users 'micah-hysong'
Job properties:
  job-id: plink-for---micah-hysong--250324-231026-37
  job-name: plink-for-prs-construction-chr19
  user-id: micah-hysong
Provider internal-id (operation): projects/1017278029283/locations/us-central1/operations/5065425374275622869
Launched job-id: plink-for---micah-hysong--250324-231026

In [12]:
!dstat --provider google-cls-v2 --project terra-vpc-sc-482f9fa9 --location us-central1 --jobs 'plink-for---micah-hysong--250324-133212-36' --users 'micah-hysong' --status '*'


Job Name         Status                                      Last Update
---------------  ------------------------------------------  --------------
plink-for-pr...  Pulling "us.gcr.io/broad-dsp-gcr-public...  03-24 13:33:10



In [2]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))




gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/:
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr10_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr10_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr11_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr11_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr12_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr12_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr13_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr13_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr14_allind.bk
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr14_allind.rds
gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/bigsnpr_backing/chr15_allind.bk
g

In [15]:
# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/dsub/logs/plink-for-prs-construction-chr2/micah-hysong/plink-for---micah-hysong--250324-133206-21-task-None.log' .")


Copying gs://fc-secure-784622b3-a3b0-42a1-a555-372ab8841bd6/dsub/logs/plink-for-prs-construction-chr2/micah-hysong/plink-for---micah-hysong--250324-133206-21-task-None.log...
/ [1 files][412.3 KiB/412.3 KiB]                                                
Operation completed over 1 objects/412.3 KiB.                                    


0